<a href="https://colab.research.google.com/github/lukalklikadze/walmart-sales-forecasting/blob/main/notebooks/model_inference.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Final model: 4-model ensemble — inference and Kaggle submission

Our final model is an **equal-weight ensemble of four pipelines**: LightGBM, Prophet,
XGBoost and N-BEATS. Averaging models from different families cancels their systematic
biases — one tends to under-forecast, another to over-forecast — and equal weights have
nothing to tune, so nothing to overfit. The combination was chosen by submitting every
sensible subset of our trained models to Kaggle and reading the board:

| candidate | public WMAE | private WMAE |
| :--- | ---: | ---: |
| **LightGBM+Prophet+XGBoost+N-BEATS** | **2559.39** | **2663.17** |
| LightGBM+Prophet+N-BEATS | 2646.52 | 2759.53 |
| LightGBM+Prophet+XGBoost | 2641.79 | 2761.82 |
| all five, 1/val-WMAE weights | 2753.74 | 2871.12 |
| all five, equal weights | 2803.89 | 2923.85 |
| LightGBM+Prophet | 2806.33 | 2958.08 |
| single best model (LightGBM) | 2846.29 | 3013.51 |

Adding TFT and DLinear always hurt (their solo validation WMAE is much weaker), so the
final ensemble uses the four strongest models only.

This notebook trains nothing. Each training notebook already logged its final pipeline
refit on the **full** train set; here we pull those four out of MLflow, run each on the
raw test frame, average, and submit.

## 1. Bootstrap

Clone the repo, read the Colab secrets, download the Kaggle data, and install what the
four members need to unpickle: LightGBM/XGBoost for the trees, `prophet`, and
`neuralforecast`+torch for N-BEATS.

In [ ]:
!pip -q install mlflow kaggle lightgbm xgboost neuralforecast prophet

!git clone -q https://github.com/lukalklikadze/walmart-sales-forecasting.git /content/repo \
    2>/dev/null || (cd /content/repo && git pull -q)
import sys; sys.path.append("/content/repo")

from src.config import setup_env, DATA_DIR, KAGGLE_COMP
tracking_uri = setup_env()
print("MLflow \u2192", tracking_uri)

import os, glob, zipfile, subprocess
os.makedirs(DATA_DIR, exist_ok=True)
subprocess.run(["kaggle","competitions","download","-c",KAGGLE_COMP,"-p",DATA_DIR,"--force"], check=True)
with zipfile.ZipFile(os.path.join(DATA_DIR, KAGGLE_COMP + ".zip")) as z: z.extractall(DATA_DIR)
for inner in glob.glob(os.path.join(DATA_DIR, "*.csv.zip")):
    with zipfile.ZipFile(inner) as z: z.extractall(DATA_DIR)

from src.data import load_raw
train, test, stores, features = load_raw()
print("train:", train.shape, "| test:", test.shape)

## 2. Imports, download settings, and wrapper classes

Two settings matter for loading models from DagsHub: multipart (chunked) downloads must be
**off** — DagsHub stalls on the parallel byte-range requests MLflow uses for big files —
and the HTTP timeout needs to be generous. The LightGBM wrapper classes are redeclared
byte-for-byte from the training notebook so cloudpickle can always resolve them.

In [ ]:
import os
# DagsHub's artifact server stalls on MLflow's multipart (parallel byte-range) downloads;
# a single plain stream works. Long timeout so big files can finish.
os.environ["MLFLOW_ENABLE_MULTIPART_DOWNLOAD"] = "false"
os.environ["MLFLOW_HTTP_REQUEST_TIMEOUT"] = "600"

import re
import numpy as np
import pandas as pd
import lightgbm as lgb
import torch
import mlflow
from contextlib import contextmanager
from mlflow.tracking import MlflowClient
from sklearn.base import BaseEstimator, RegressorMixin
from src.features import CalendarFeatures, LagFeatures, DropColumns

CATEGORICAL_COLS = ["Store", "Dept"]

def signed_log1p(y):
    return np.sign(y) * np.log1p(np.abs(y))

def signed_expm1(z):
    return np.sign(z) * np.expm1(np.abs(z))

def holiday_weights(is_holiday):
    return np.where(np.asarray(is_holiday) == 1, 5.0, 1.0)

def to_lgb_frame(df, train_categories=None):
    y = df["Weekly_Sales"].to_numpy(dtype=float) if "Weekly_Sales" in df.columns else None
    X = df.drop(columns=["Weekly_Sales"]).copy() if "Weekly_Sales" in df.columns else df.copy()
    w = holiday_weights(df["IsHoliday"])
    for col in CATEGORICAL_COLS:
        if train_categories is None:
            X[col] = X[col].astype("category")
        else:
            X[col] = pd.Categorical(X[col], categories=train_categories[col])
    return X, y, w

class ToLGBFrame(BaseEstimator):
    def fit(self, X, y=None):
        X_lgb, _, _ = to_lgb_frame(X)
        self.train_categories_ = {c: X_lgb[c].cat.categories for c in CATEGORICAL_COLS}
        return self
    def transform(self, X):
        X_lgb, _, _ = to_lgb_frame(X, train_categories=self.train_categories_)
        return X_lgb

class LGBMBoosterRegressor(BaseEstimator, RegressorMixin):
    def __init__(self, num_boost_round=100, **lgb_params):
        self.num_boost_round = num_boost_round
        self.lgb_params = lgb_params
    def fit(self, X, y, sample_weight=None):
        train_set = lgb.Dataset(X, label=y, weight=sample_weight,
                                categorical_feature=CATEGORICAL_COLS, free_raw_data=False)
        self.booster_ = lgb.train(self.lgb_params, train_set, num_boost_round=self.num_boost_round)
        return self
    def predict(self, X):
        return self.booster_.predict(X, num_iteration=self.booster_.best_iteration)

## 3. Load the four ensemble members from MLflow

Each member's training notebook logged a final full-train pipeline. MLflow 3.x stores it
as a standalone *logged model*, so we resolve the real model id, register a version under
a stable name, and load through the registry (the path that works on DagsHub).

Three compatibility fixes for the N-BEATS pipeline, all no-ops for the others:

1. it was pickled on a **GPU**, so on a CPU runtime every tensor is mapped to CPU while
   unpickling and the saved trainer settings are forced to CPU;
2. it was saved with an **older neuralforecast** — attributes the new library expects
   (categorical-exogenous support our univariate model never used) are filled with empty
   defaults on demand;
3. predictions run through `robust_predict`, which applies fix 2 automatically and retries.

Inference is only valid with **all four** members, so this cell fails loudly if any is
missing rather than silently averaging fewer models.

In [ ]:
client = MlflowClient()

# the four ensemble members -> (experiment, runName filter)
REGISTRY = {
    "LightGBM": ("lightGBM_Training", "tags.mlflow.runName LIKE 'final_full_train__%'"),
    "Prophet":  ("Prophet_Training",  "tags.mlflow.runName LIKE 'final_full_train__%'"),
    "XGBoost":  ("XGBoost_Training",  "tags.mlflow.runName LIKE 'final_full_train__%'"),
    "N-BEATS":  ("NBEATS_Training",   "tags.mlflow.runName LIKE 'final_full_train__%'"),
}

@contextmanager
def cpu_unpickle():
    """On CPU-only runtimes, map GPU-pickled tensors to CPU while unpickling."""
    if torch.cuda.is_available():
        yield; return
    orig = torch.load
    torch.load = lambda *a, **k: orig(*a, **{**k, "map_location": "cpu"})
    try:
        yield
    finally:
        torch.load = orig

def force_cpu(pipe):
    """Point every saved device hint inside a pipeline at CPU (no-op for non-torch models)."""
    for step in pipe.named_steps.values():
        if hasattr(step, "device"):
            step.device = "cpu"
        nf = getattr(step, "nf_", None)
        if nf is not None:
            for m in nf.models:
                tk = getattr(m, "trainer_kwargs", None)
                if isinstance(tk, dict):
                    tk["accelerator"] = "cpu"; tk["devices"] = 1; tk.pop("strategy", None)
        mdl = getattr(step, "model_", None)
        if mdl is not None and hasattr(mdl, "to"):
            mdl.to("cpu")
    return pipe

def robust_predict(pipe, X, max_shims=12):
    """predict() that fills library-version attribute gaps with safe empty defaults.
    Our N-BEATS is univariate, so empty exog lists reproduce the old behaviour exactly."""
    for _ in range(max_shims):
        try:
            return np.asarray(pipe.predict(X), dtype=float)
        except AttributeError as e:
            mo = re.search(r"'(\w+)' object has no attribute '(\w+)'", str(e))
            if not mo:
                raise
            cls, attr = mo.groups()
            default = [] if attr.endswith("_list") else ({} if "vocab" in attr else None)
            targets = []
            for step in pipe.named_steps.values():
                nf = getattr(step, "nf_", None)
                if nf is not None:
                    targets = [nf] if cls == type(nf).__name__ else list(nf.models)
            if not targets:
                raise
            for t in targets:
                if not hasattr(t, attr):
                    setattr(t, attr, default)
            print(f"  shimmed {cls}.{attr} = {default!r}", flush=True)
    raise RuntimeError("still failing after max_shims attribute fixes")

def _val_wmae(run):
    for src in (run.data.params, run.data.metrics):
        for k in ("val_wmae_of_chosen_cfg", "val_wmae"):
            if k in src:
                try: return float(src[k])
                except (TypeError, ValueError): pass
    return float("nan")

def _resolve_logged_model(run):
    """MLflow 3 stores log_model() output as a standalone logged model, NOT under
    runs:/<id>/model. Find its real id via run.outputs, else search the experiment."""
    model_id = None
    try:
        mo = run.outputs.model_outputs
        if mo: model_id = mo[0].model_id
    except Exception:
        pass
    if model_id is None:
        try:
            lms = client.search_logged_models(experiment_ids=[run.info.experiment_id])
            cand = [m for m in lms if getattr(m, "source_run_id", None) == run.info.run_id]
            if cand: model_id = cand[0].model_id
        except Exception:
            pass
    if model_id is None:
        return None, None
    return model_id, client.get_logged_model(model_id)

def load_final_pipeline(name, experiment, run_filter):
    exp = client.get_experiment_by_name(experiment)
    assert exp is not None, f"experiment {experiment!r} not found"
    runs = client.search_runs([exp.experiment_id], filter_string=run_filter,
                              order_by=["attributes.start_time DESC"], max_results=1)
    assert runs, f"no run matching {run_filter} in {experiment}"
    run = runs[0]
    model_id, lm = _resolve_logged_model(run)
    uris = []
    if lm is not None:
        uris += [f"models:/{model_id}", lm.artifact_location]
        reg_name = f"walmart-ens-{name}"   # register at the real location, load via registry
        try:
            try: client.get_registered_model(reg_name)
            except Exception: client.create_registered_model(reg_name)
            mv = client.create_model_version(name=reg_name, source=lm.artifact_location,
                                             run_id=run.info.run_id)
            uris.append(f"models:/{reg_name}/{mv.version}")
        except Exception:
            pass
    uris.append(f"runs:/{run.info.run_id}/model")
    last = None
    for uri in uris:
        try:
            with cpu_unpickle():
                pipe = mlflow.sklearn.load_model(uri)
            if not torch.cuda.is_available():
                force_cpu(pipe)
            return pipe, _val_wmae(run), run.info.run_id
        except Exception as e:
            last = e
    raise RuntimeError(f"all load URIs failed; last error: {last}")

preds = {}
for name, (experiment, run_filter) in REGISTRY.items():
    for attempt in (1, 2):   # DagsHub downloads are flaky; retry once per model
        print(f"loading {name} (attempt {attempt}) ...", flush=True)
        try:
            pipe, vw, run_id = load_final_pipeline(name, experiment, run_filter)
            p = robust_predict(pipe, test)
            assert len(p) == len(test) and np.isfinite(p).all(), "bad predictions"
            preds[name] = p
            print(f"[ok] {name:9s} val WMAE {vw:8.1f}  run {run_id[:8]}  sample {p[:2].round(1)}", flush=True)
            break
        except Exception as e:
            if attempt == 2:
                raise RuntimeError(f"{name} failed to load twice: {e}") from e

missing = set(REGISTRY) - set(preds)
assert not missing, f"ensemble members missing: {missing}"
print("\nall four members loaded")

## 4. Average and build the submission

The final prediction is the plain mean of the four members, floored at zero (sales are
non-negative). Kaggle wants `Id` as `Store_Dept_Date` plus `Weekly_Sales`; the pipelines
keep row order, so the ids come straight from the test frame.

In [ ]:
ensemble_preds = np.clip(np.mean([preds[m] for m in REGISTRY], axis=0), 0, None)

submission = pd.DataFrame({
    "Id": (test["Store"].astype(str) + "_" + test["Dept"].astype(str) + "_" +
           test["Date"].dt.strftime("%Y-%m-%d")),
    "Weekly_Sales": ensemble_preds,
})

sample = pd.read_csv(os.path.join(DATA_DIR, "sampleSubmission.csv"))
assert len(submission) == len(sample)
assert set(submission["Id"]) == set(sample["Id"]), "submission ids do not match the sample"

SUBMISSION_PATH = os.path.join(DATA_DIR, "submission.csv")
submission.to_csv(SUBMISSION_PATH, index=False)
print("wrote", SUBMISSION_PATH, submission.shape)
submission.head()

## 5. Submit to Kaggle

Uploads the file and prints the board. Expected scores for this ensemble:
**Public 2559.39 / Private 2663.17**. Rerun the cell if a row still says PENDING.

In [ ]:
MESSAGE = "final model: equal-weight ensemble of LightGBM+Prophet+XGBoost+N-BEATS"
subprocess.run(["kaggle", "competitions", "submit",
                "-c", KAGGLE_COMP, "-f", SUBMISSION_PATH, "-m", MESSAGE], check=True)

import time; time.sleep(60)
r = subprocess.run(["kaggle", "competitions", "submissions", "-c", KAGGLE_COMP],
                   capture_output=True, text=True)
print(r.stdout)